# 学習曲線 — SmallUNet 系バリアント

全 fold × 全サイズを一括実行する。既存の結果はスキップされる。

**Colab Pro GPU ランタイム推奨**（全30ジョブで1〜2時間程度）。

| バリアント例 | BACKBONE | AUGMENT | LOSS | 目的 |
|---|---|---|---|---|
| `smallunet_aug0_mse_s15` (デフォルト) | smallunet | False | mse | スクラッチ baseline |
| `smallunet_aug1_awl_s15` | smallunet | True | awl | 拡張+AWLのみの効果確認 |

In [ ]:
# ===== 設定（ここを変えて実行する）=====
FOLDS = [1, 2, 3, 4, 5]            # 実行するfold（途中再開も可）
SIZES = [15, 30, 60, 90, 120, 149]

BACKBONE = "smallunet"  # "smallunet" 固定（変える場合は resnet34/01_train.ipynb を使う）
AUGMENT  = False        # データ拡張
LOSS     = "mse"        # "mse" or "awl"
SIGMA    = 15.0         # ヒートマップのσ（px）
# ========================================

DRIVE_DATA_DIR = "/content/drive/MyDrive/anglist_data"
DRIVE_LC_DIR   = "/content/drive/MyDrive/anglist_learning_curve"
EPOCHS     = 100
BATCH_SIZE = 4
RESIZE     = [512, 512]
LR         = 1e-3  # SmallUNet はスクラッチ学習のため高めのLRを使用

VARIANT = f"{BACKBONE}_aug{int(AUGMENT)}_{LOSS}_s{int(SIGMA)}"
print(f"variant: {VARIANT}  LR: {LR}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

REPO = "/content/anglist"
DATASET_PY = f"{REPO}/train/dataset.py"

# dataset.py が存在しない = 壊れた clone → 削除して再取得
if os.path.exists(REPO) and not os.path.exists(DATASET_PY):
    print("壊れた clone を検出 → 削除して再クローン")
    shutil.rmtree(REPO)

if os.path.exists(REPO):
    print("git pull...")
    os.system("cd /content/anglist && git pull")
else:
    print("git clone...")
    ret = os.system("git clone https://github.com/masaki39/anglist /content/anglist")
    if ret != 0:
        raise RuntimeError("git clone 失敗")

assert os.path.exists(DATASET_PY), f"clone 後も dataset.py が見つかりません: {DATASET_PY}"
print("✓ clone OK:", DATASET_PY)

os.system("pip install -q tqdm onnx onnxruntime onnxscript 'albumentations>=1.3,<2.0' segmentation-models-pytorch")


In [ ]:
import json, math, os, random, shutil, subprocess, sys
import numpy as np
import onnx
import onnxruntime as ort
import torch

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

sys.path.insert(0, "/content/anglist/train")
from dataset import LANDMARK_ORDER, _percentile_clip_norm, _resize_with_padding

with open(os.path.join(DRIVE_LC_DIR, "folds.json")) as f:
    fold_data = json.load(f)

def preprocess(img_np, resize=(512, 512)):
    if img_np.ndim == 3: img_np = img_np[0]
    img_np = _percentile_clip_norm(img_np)
    t = torch.from_numpy(img_np).unsqueeze(0)
    t, scale, pad_x, pad_y = _resize_with_padding(t, resize)
    return t.unsqueeze(0), scale, pad_x, pad_y

def postprocess(hm):
    hm = hm[0]
    return [(float(np.unravel_index(np.argmax(c), c.shape)[1]),
             float(np.unravel_index(np.argmax(c), c.shape)[0])) for c in hm]

def run_subprocess(cmd, cwd):
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        print("=== STDOUT ==="); print(result.stdout[-5000:])
        print("=== STDERR ==="); print(result.stderr[-5000:])
        cmd_str = " ".join(cmd)
        raise RuntimeError(f"Command failed (exit {result.returncode}): {cmd_str}")
    lines = (result.stdout + result.stderr).strip().split("\n")
    print("\n".join(lines[-5:]))

def _angle_between(v1, v2):
    import math
    dot = v1[0]*v2[0] + v1[1]*v2[1]
    l1 = math.hypot(*v1); l2 = math.hypot(*v2)
    if l1 == 0 or l2 == 0: return None
    return math.degrees(math.acos(max(-1.0, min(1.0, dot/(l1*l2)))))

def _slope_angle(v):
    import math
    ang = math.degrees(math.atan2(v[1], v[0]))
    if ang > 90: ang -= 180
    elif ang < -90: ang += 180
    return -ang

def _vert_angle(v):
    import math
    ang = math.degrees(math.atan2(v[0], -v[1]))
    if ang > 90: ang -= 180
    elif ang < -90: ang += 180
    return ang

def _wrap(a):
    while a > 180: a -= 360
    while a < -180: a += 360
    return a

def compute_angles_mm(pts):
    """pts: dict name -> (x_mm, y_mm). Returns angle dict or None."""
    try:
        required = ["L1_ant", "L1_post", "S1_ant", "S1_post", "FH"]
        if not all(k in pts for k in required): return None
        FH = pts["FH"]; S1a = pts["S1_ant"]; S1p = pts["S1_post"]
        L1a = pts["L1_ant"]; L1p = pts["L1_post"]
        v_S1 = (S1p[0]-S1a[0], S1p[1]-S1a[1])
        v_L1 = (L1p[0]-L1a[0], L1p[1]-L1a[1])
        S1m = ((S1a[0]+S1p[0])/2, (S1a[1]+S1p[1])/2)
        v_pel = (S1m[0]-FH[0], S1m[1]-FH[1])
        theta = _angle_between(v_pel, v_S1)
        if theta is None: return None
        return {"PI": abs(90.0 - theta), "PT": _vert_angle(v_pel),
                "SS": _slope_angle(v_S1), "LL": _wrap(_slope_angle(v_S1) - _slope_angle(v_L1))}
    except Exception:
        return None


In [ ]:
results_dir = os.path.join(DRIVE_LC_DIR, "results", VARIANT)
os.makedirs(results_dir, exist_ok=True)

for FOLD in FOLDS:
    print("=" * 60)
    print(f"FOLD {FOLD}  ({FOLDS.index(FOLD)+1}/{len(FOLDS)})  VARIANT={VARIANT}")
    print("=" * 60)

    test_ids      = fold_data["folds"][str(FOLD)]
    all_train_ids = [cid for k, ids in fold_data["folds"].items()
                     for cid in ids if k != str(FOLD)]

    LOCAL_TEST = f"/content/test_data_fold{FOLD}"
    if os.path.exists(LOCAL_TEST): shutil.rmtree(LOCAL_TEST)
    os.makedirs(LOCAL_TEST)
    for cid in test_ids:
        for ext in ["_image.npy", "_landmarks.json"]:
            shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TEST)
    print(f"  test={len(test_ids)} samples")

    for SIZE in SIZES:
        out_path = os.path.join(results_dir, f"fold{FOLD}_size{SIZE:03d}.json")
        if os.path.exists(out_path):
            print(f"[skip] fold{FOLD}_size{SIZE:03d}")
            continue

        print(f"--- FOLD={FOLD}  SIZE={SIZE} ---")

        random.seed(FOLD * 1000 + SIZE)
        train_ids = sorted(all_train_ids)
        random.shuffle(train_ids)
        train_ids = train_ids[:SIZE]

        LOCAL_TRAIN = "/content/train_data"
        if os.path.exists(LOCAL_TRAIN): shutil.rmtree(LOCAL_TRAIN)
        os.makedirs(LOCAL_TRAIN)
        for cid in train_ids:
            for ext in ["_image.npy", "_landmarks.json"]:
                shutil.copy(os.path.join(DRIVE_DATA_DIR, cid + ext), LOCAL_TRAIN)
        print(f"  train={len(train_ids)}")

        SAVE_DIR = f"/content/runs/{VARIANT}/fold{FOLD}_size{SIZE:03d}"
        cmd = [
            "python", "train.py",
            "--data-dir", LOCAL_TRAIN,
            "--save-dir", SAVE_DIR,
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--sigma", str(SIGMA),
            "--resize", str(RESIZE[0]), str(RESIZE[1]),
            "--backbone", BACKBONE,
            "--loss", LOSS,
        ]
        if AUGMENT:
            cmd.append("--augment")
        run_subprocess(cmd, cwd="/content/anglist/train")

        run_subprocess([
            "python", "export_onnx.py",
            "--checkpoint", f"{SAVE_DIR}/best.pt",
            "--output", "/content/model_raw.onnx",
            "--height", str(RESIZE[0]),
            "--width",  str(RESIZE[1]),
        ], cwd="/content/anglist/train")

        proto = onnx.load("/content/model_raw.onnx", load_external_data=True)
        onnx.save_model(proto, "/content/model.onnx", save_as_external_data=False)

        sess = ort.InferenceSession("/content/model.onnx",
                                    providers=["CPUExecutionProvider"])
        all_errors = {name: [] for name in LANDMARK_ORDER}

        per_case_mres = []
        angle_case_errors = {"PI": [], "PT": [], "SS": [], "LL": []}
        for cid in test_ids:
            img_np = np.load(os.path.join(LOCAL_TEST, cid + "_image.npy"))
            inp_t, scale, pad_x, pad_y = preprocess(img_np)
            ort_out = sess.run(None, {"image": inp_t.numpy()})
            pred = postprocess(ort_out[0])

            with open(os.path.join(LOCAL_TEST, cid + "_landmarks.json")) as fp:
                meta = json.load(fp)
            spacing = meta["metadata"]["spacing"][0]
            lm = meta["landmarks_ijk"]

            for (px, py), name in zip(pred, LANDMARK_ORDER):
                gx, gy = lm[name]["i"], lm[name]["j"]
                x_o = (px - pad_x) / scale
                y_o = (py - pad_y) / scale
                all_errors[name].append(
                    math.sqrt((x_o - gx)**2 + (y_o - gy)**2) * spacing)
            # --- per-case angle error & outlier tracking ---
            pred_pts_mm = {}
            for (px2, py2), nm in zip(pred, LANDMARK_ORDER):
                xo = (px2 - pad_x) / scale
                yo = (py2 - pad_y) / scale
                pred_pts_mm[nm] = (xo * spacing, yo * spacing)
            gt_pts_mm = {nm: (lm[nm]["i"] * spacing, lm[nm]["j"] * spacing)
                         for nm in LANDMARK_ORDER}
            case_mre = sum(all_errors[nm][-1] for nm in LANDMARK_ORDER) / len(LANDMARK_ORDER)
            per_case_mres.append(case_mre)
            gt_ang = compute_angles_mm(gt_pts_mm)
            pred_ang = compute_angles_mm(pred_pts_mm)
            if gt_ang and pred_ang:
                for _a in ["PI", "PT", "SS", "LL"]:
                    angle_case_errors[_a].append(abs(gt_ang[_a] - pred_ang[_a]))

        # --- outlier detection (case MRE > 3x median) ---
        if per_case_mres:
            _sorted = sorted(per_case_mres)
            _med = _sorted[len(_sorted)//2]
            _thr = 3 * _med
            ok_idx = [i for i, m in enumerate(per_case_mres) if m <= _thr]
        else:
            ok_idx = list(range(len(test_ids)))
        n_outliers = len(test_ids) - len(ok_idx)

        results = {"fold": FOLD, "size": SIZE, "variant": VARIANT,
                   "n_test": len(test_ids), "landmarks": {}}
        all_vals = []
        for name, errs in all_errors.items():
            mre  = sum(errs) / len(errs)
            sdr2 = sum(1 for e in errs if e <= 2.0) / len(errs) * 100
            sdr4 = sum(1 for e in errs if e <= 4.0) / len(errs) * 100
            results["landmarks"][name] = {"mre_mm": mre, "sdr2": sdr2, "sdr4": sdr4}
            all_vals.extend(errs)
        _angle_mae = {a: sum(v)/len(v) for a, v in angle_case_errors.items() if v}
        results["overall"] = {
            "mre_mm": sum(all_vals) / len(all_vals),
            "sdr2":   sum(1 for e in all_vals if e <= 2.0) / len(all_vals) * 100,
            "sdr4":   sum(1 for e in all_vals if e <= 4.0) / len(all_vals) * 100,
            "angle_mae": _angle_mae,
        }
        _excl_vals = [v for i, nm in enumerate(LANDMARK_ORDER) for j, v in enumerate(all_errors[nm]) if j in ok_idx]
        _excl_ang = {a: sum(v[i] for i in ok_idx if i < len(v))/len([i for i in ok_idx if i < len(v)])
                     for a, v in angle_case_errors.items() if any(i < len(v) for i in ok_idx)}
        results["overall_excl_outliers"] = {
            "n": len(ok_idx), "n_outliers": n_outliers,
            "mre_mm": sum(_excl_vals)/len(_excl_vals) if _excl_vals else None,
            "angle_mae": _excl_ang,
        }
        results["per_case_mre_mm"] = per_case_mres
        with open(out_path, "w") as fp:
            json.dump(results, fp, indent=2)
        mre_v = results["overall"]["mre_mm"]
        sdr_v = results["overall"]["sdr4"]
        print(f"  MRE={mre_v:.2f}mm  SDR@4mm={sdr_v:.1f}%  -> saved")

print("全fold・全サイズ完了")